# E0-rerun: UNet Raw BTXRD Tumor-Only 224x224

Rerun raw BTXRD with the new repo pipeline, seed42 split, UNet baseline, and the same E1a-v2 loss setting (`positive_weight=5`). This run is for a fairer raw-vs-preprocessed comparison.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/lehngoc/BTXRD-LViT.git"
BRANCH = "model/e1-unet-baseline"
REPO_ROOT = Path("/kaggle/working/BTXRD-LViT")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f"Repo already exists: {REPO_ROOT}")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
import torch
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Paths

`DATA_ROOT` must contain raw images, raw masks converted from LabelMe, and the seed42 split CSVs:

- `data/raw/images/`
- `data/processed/masks/`
- `data/exports/btxrd_preprocessed/train.csv`
- `data/exports/btxrd_preprocessed/val.csv`
- `data/exports/btxrd_preprocessed/test.csv`

In [ ]:
from pathlib import Path

# Change this if your Kaggle Dataset path is different.
DATA_ROOT = Path("/kaggle/input/datasets/lehngoc/btxrd-preprocessed-dataset/btxrd-preprocessed")

# If you uploaded a new raw/full dataset, common examples are:
# DATA_ROOT = Path("/kaggle/input/btxrd-full")
# DATA_ROOT = Path("/kaggle/input/btxrd-raw-seed42")

OUTPUT_DIR = Path("/kaggle/working/experiments/E0_rerun_unet_raw_224_tumor_only_pos5")
MANIFEST_DIR = Path("/kaggle/working/e0_rerun_raw_manifests")
RUNTIME_CONFIG = Path("/kaggle/working/e0_rerun_unet_raw_tumor_only_kaggle.yaml")

IMAGE_SIZE = 224
BATCH_SIZE = 4
NUM_WORKERS = 2
EPOCHS = 200
LR = 3e-4
POSITIVE_WEIGHT = 5.0
PATIENCE = 40

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("MANIFEST_DIR =", MANIFEST_DIR)

In [ ]:
required = [
    DATA_ROOT / "data/raw/images",
    DATA_ROOT / "data/processed/masks",
    DATA_ROOT / "data/exports/btxrd_preprocessed/train.csv",
    DATA_ROOT / "data/exports/btxrd_preprocessed/val.csv",
    DATA_ROOT / "data/exports/btxrd_preprocessed/test.csv",
]

for path in required:
    print(path, "->", path.exists())

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required raw E0-rerun inputs:\n" + "\n".join(missing))

## Build Raw Manifests

The split membership comes from the existing seed42 exported CSVs. Only `image_path` and `mask_path` are rewritten to raw image and raw converted mask paths.

In [ ]:
import pandas as pd

RAW_IMAGE_DIR = DATA_ROOT / "data/raw/images"
RAW_MASK_DIR = DATA_ROOT / "data/processed/masks"

def find_relative_raw_image(image_id: str) -> str:
    stem = Path(str(image_id)).stem
    candidates = [
        Path("data/raw/images") / str(image_id),
        Path("data/raw/images") / f"{stem}.jpeg",
        Path("data/raw/images") / f"{stem}.jpg",
        Path("data/raw/images") / f"{stem}.png",
    ]
    for rel_path in candidates:
        if (DATA_ROOT / rel_path).exists():
            return str(rel_path)
    raise FileNotFoundError(f"Missing raw image for {image_id}")

def find_relative_raw_mask(image_id: str) -> str:
    stem = Path(str(image_id)).stem
    candidates = [
        Path("data/processed/masks") / f"{stem}.png",
        Path("data/processed/masks") / f"{stem}.jpg",
        Path("data/processed/masks") / f"{stem}.jpeg",
    ]
    for rel_path in candidates:
        if (DATA_ROOT / rel_path).exists():
            return str(rel_path)
    raise FileNotFoundError(f"Missing raw mask for {image_id}")

def write_raw_manifest(split: str) -> Path:
    src_csv = DATA_ROOT / f"data/exports/btxrd_preprocessed/{split}.csv"
    dst_csv = MANIFEST_DIR / f"{split}.csv"
    df = pd.read_csv(src_csv)
    df["image_path"] = df["image_id"].map(find_relative_raw_image)
    df["mask_path"] = df["image_id"].map(find_relative_raw_mask)
    df.to_csv(dst_csv, index=False)
    tumor_count = int(df["tumor"].astype(int).sum())
    print(f"{split}: rows={len(df)} tumor={tumor_count} normal={len(df)-tumor_count} -> {dst_csv}")
    return dst_csv

train_csv = write_raw_manifest("train")
val_csv = write_raw_manifest("val")
test_csv = write_raw_manifest("test")

pd.read_csv(train_csv).head()

In [ ]:
import yaml

cfg = {
    "experiment": {"name": "E0_rerun_unet_raw_224_tumor_only_pos5"},
    "data": {
        "root_dir": str(DATA_ROOT),
        "tumor_only": True,
        "train_csv": str(train_csv),
        "val_csv": str(val_csv),
        "test_csv": str(test_csv),
    },
    "model": {
        "name": "unet",
        "in_channels": 3,
        "out_channels": 1,
        "base_channels": 32,
    },
    "training": {
        "seed": 42,
        "device": "cuda",
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "weight_decay": 1e-4,
        "bce_weight": 1.0,
        "dice_weight": 1.0,
        "positive_weight": POSITIVE_WEIGHT,
        "dice_on_tumor_only": False,
        "early_stopping_patience": PATIENCE,
        "label_fraction": 1.0,
        "text_column": "text_lvit_prompt",
        "output_dir": str(OUTPUT_DIR),
    },
    "metrics": {
        "threshold": 0.5,
        "min_fp_area_ratio": 0.001,
    },
}

RUNTIME_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(RUNTIME_CONFIG.read_text())

In [ ]:
!python src/training/smoke_test_model_pipeline.py --config {RUNTIME_CONFIG} --samples-per-split 8

In [ ]:
!python src/training/train_unet.py --config {RUNTIME_CONFIG} --device cuda

In [ ]:
best_ckpt = OUTPUT_DIR / "best.pt"
assert best_ckpt.exists(), best_ckpt

!python src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split val --device cuda --output {OUTPUT_DIR / 'val_metrics.json'}
!python src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split test --device cuda --output {OUTPUT_DIR / 'test_metrics.json'}

In [ ]:
import json

for name in ["best_summary.json", "val_metrics.json", "test_metrics.json"]:
    path = OUTPUT_DIR / name
    print("\n===", name, "===")
    print(json.dumps(json.loads(path.read_text()), indent=2))

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
sweep = []
for thr in thresholds:
    out = OUTPUT_DIR / f"val_metrics_thr{int(thr * 100):02d}.json"
    !python src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split val --device cuda --threshold {thr} --output {out}
    metrics = json.loads(out.read_text())
    sweep.append({
        "threshold": thr,
        "tumor_dice": metrics["tumor_dice"],
        "tumor_iou": metrics["tumor_iou"],
        "normal_pred_area_ratio": metrics["normal_pred_area_ratio"],
        "normal_fp_image_rate": metrics["normal_fp_image_rate"],
    })

(OUTPUT_DIR / "threshold_sweep_metrics.json").write_text(json.dumps(sweep, indent=2), encoding="utf-8")
print(json.dumps(sweep, indent=2))

In [ ]:
!python src/training/visualize_unet_predictions.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split val --device cuda --output-dir {OUTPUT_DIR / 'visual_checks_val'} --max-tumor 24 --max-normal 0
!python src/training/visualize_unet_predictions.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split test --device cuda --output-dir {OUTPUT_DIR / 'visual_checks_test'} --max-tumor 24 --max-normal 0

In [ ]:
import zipfile

metrics_zip = Path("/kaggle/working/E0_rerun_raw_tumor_only_pos5_metrics_only.zip")
wanted = [
    "history.csv",
    "best_summary.json",
    "val_metrics.json",
    "test_metrics.json",
    "threshold_sweep_metrics.json",
    "config.json",
]
with zipfile.ZipFile(metrics_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in wanted:
        path = OUTPUT_DIR / name
        if path.exists():
            z.write(path, arcname=name)
    for path in sorted(OUTPUT_DIR.glob("val_metrics_thr*.json")):
        z.write(path, arcname=path.name)

full_zip = Path("/kaggle/working/E0_rerun_raw_tumor_only_pos5_full_artifacts.zip")
with zipfile.ZipFile(full_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            z.write(path, arcname=str(path.relative_to(OUTPUT_DIR)))

print("metrics zip:", metrics_zip, metrics_zip.stat().st_size / 1024 / 1024, "MB")
print("full zip:", full_zip, full_zip.stat().st_size / 1024 / 1024, "MB")